In [1]:
# Jupyter cell — VS Code

from pathlib import Path
import re
import pandas as pd

# ---------- Paths ----------
SAMPLE_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10.csv")
OUT_XLSX   = SAMPLE_CSV.with_name("stratified_sample_moe10_output.xlsx")

CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files")

# ---------- Helpers ----------
YAML_EXTS = {".yml", ".yaml"}
GRADLE_EXTS = {".gradle"}  # .gradle.kts handled via name.endswith

def truth01(val):
    s = pd.Series([val])
    if s.dtype == bool:
        return int(s.iloc[0])
    t = s.astype(str).str.strip().str.lower()
    return int(t.iloc[0] in {"1","true","t","yes","y","on"})

def safe_read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return ""

def repo_key_from_filename(fname: str) -> str:
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()

def index_repo_files(root: Path):
    idx = {}
    if root.exists():
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            is_yaml   = p.suffix.lower() in YAML_EXTS
            is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")
            if is_yaml or is_gradle:
                key = repo_key_from_filename(p.name)
                idx.setdefault(key, []).append(p)
    return idx

def build_androidtest_presence_index(tests_dir: Path):
    present = set()
    if tests_dir.exists():
        for p in tests_dir.iterdir():
            if p.is_file():
                present.add(repo_key_from_filename(p.name))
    return present

# --- succinct patterns ---
YAML_SIGNAL_PATTERNS = {
    "reactivecircus_runner": r"uses:\s*reactivecircus/android-emulator-runner@",
    "malinskiy_runner":      r"uses:\s*malinskiy/action-android/emulator-run-cmd@",
    "gradle_connected":      r"\b(?:^|[^\w])gradle[w]?\b[^\n\r]*\bconnected(androidtest|check)\b",
    "device_or_managed":     r"\b(?:manageddevice|devicecheck|alldevicechecks)\b",
    "variant_androidtest":   r"\b[a-zA-Z0-9_]+(?:Debug|Release)?AndroidTest\b",
    "adb_instrument":        r"\badb\s+(?:-s\s+\S+\s+)?shell\s+am\s+instrument\b",
    "gcloud_ftl":            r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    "emulator_launch":       r"\bemulator\b[^\n\r]*(?:-avd\s+\S+|@\S+)",
    "avdmanager":            r"\bavdmanager\b|\bandroid\s+create\s+avd\b",
    "sdk_sysimg":            r"\bsdkmanager\b[^\n\r]*system-images;android-",
    "flutter_android":       r"\bflutter\s+(?:drive|test)\b[^\n\r]*(?:-d\s+(?:android|emulator-\d+))",
}
BUILD_SIGNAL_PATTERNS = {
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    "androidTestDependency_str": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.:-]+\s*\)",
    "androidx_test":           r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso":                r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator":             r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator":            r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark":               r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    "gmd_block_hint":          r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices",
    "useOrchestrator_true":    r"\buseOrchestrator\b\s*(=|\s)\s*true\b",
    "connectedAndroidTest":    r"\bconnectedAndroidTest\b",
}
def any_hit(text: str, patterns: dict) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE) for p in patterns.values())

# ---------- Load sample & build indices ----------
df = pd.read_csv(SAMPLE_CSV, low_memory=False)
if "full_name" not in df.columns:
    raise KeyError("Sample must contain a 'full_name' column.")

cfg_index = index_repo_files(CONFIG_DIR)
at_present = build_androidtest_presence_index(TESTS_DIR)

# ---------- Scan per unique repo key, then map back to rows ----------
keys = df["full_name"].astype(str).str.strip().str.lower()
uniq = keys.unique()

scan_result = {}
for k in uniq:
    yaml_found = 0
    build_found = 0
    for p in cfg_index.get(k, []):
        t = safe_read_text(p)
        if not t:
            continue
        is_yaml   = p.suffix.lower() in YAML_EXTS
        is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")
        if is_yaml and not yaml_found and any_hit(t, YAML_SIGNAL_PATTERNS):
            yaml_found = 1
        if is_gradle and not build_found and any_hit(t, BUILD_SIGNAL_PATTERNS):
            build_found = 1
        if yaml_found and build_found:
            break
    at_found = 1 if k in at_present else 0
    scan_result[k] = (yaml_found, build_found, at_found)

df["YAML_Check"]  = [scan_result.get(k, (0,0,0))[0] for k in keys]
df["Build_Check"] = [scan_result.get(k, (0,0,0))[1] for k in keys]
df["AT_Check"]    = [scan_result.get(k, (0,0,0))[2] for k in keys]

# ---------- Note: only when detection differs from predictions ----------
notes = []
has_yaml_pred  = "YAML_pred"  in df.columns
has_build_pred = "Build_pred" in df.columns
has_at_pred    = "AT_pred"    in df.columns

for i, row in df.iterrows():
    mismatches = []
    if has_yaml_pred:
        yp = truth01(row["YAML_pred"])
        if yp != int(row["YAML_Check"]):
            mismatches.append(f"YAML mismatch (pred={yp}, found={int(row['YAML_Check'])})")
    if has_build_pred:
        bp = truth01(row["Build_pred"])
        if bp != int(row["Build_Check"]):
            mismatches.append(f"Build mismatch (pred={bp}, found={int(row['Build_Check'])})")
    if has_at_pred:
        ap = truth01(row["AT_pred"])
        if ap != int(row["AT_Check"]):
            mismatches.append(f"AT mismatch (pred={ap}, found={int(row['AT_Check'])})")
    notes.append("; ".join(mismatches))

df["Note"] = notes

# ---------- Save Excel next to input; original cols + new check cols ----------
# Put the 4 new columns at the end
new_cols = ["YAML_Check", "Build_Check", "AT_Check", "Note"]
ordered_cols = list(df.columns)  # existing order already has new cols at end
df.to_excel(OUT_XLSX, index=False)
print(f"[DONE] Wrote: {OUT_XLSX}")


[DONE] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx


## accuracy check

In [6]:
# %% [markdown]
# Per-stratum accuracy + per-factor PR/F1 + overall (micro/macro) PR/F1
# Input:  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx
# Output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- CONFIG ----------
FOLDER = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final")
IN_XLSX = FOLDER / "stratified_sample_moe10_output.xlsx"
OUT_XLSX = FOLDER / "validation_metrics.xlsx"

# ---------- Helpers ----------
def to01(s: pd.Series) -> pd.Series:
    """Robust boolean -> {0,1}."""
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)
    t = s.astype(str).str.strip().str.lower()
    truthy = {"1","true","t","yes","y","on"}
    falsy  = {"0","false","f","no","n","off","","none","null","nan"}
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[t.isin(truthy)] = 1
    out[t.isin(falsy)]  = 0
    # fallback numeric
    num = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+","", regex=True), errors="coerce")
    out = out.where(out.notna(), (num.fillna(0) != 0).astype(int))
    return out.astype(int)

def pr_from_counts(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec  = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1   = (2*prec*rec)/(prec+rec) if (pd.notna(prec) and pd.notna(rec) and (prec+rec)>0) else np.nan
    return prec, rec, f1

def counts_for(pred, true):
    tp = int(((pred==1) & (true==1)).sum())
    fp = int(((pred==1) & (true==0)).sum())
    fn = int(((pred==0) & (true==1)).sum())
    tn = int(((pred==0) & (true==0)).sum())
    return tp, fp, fn, tn

# ---------- Load ----------
df = pd.read_excel(IN_XLSX)

need = ["YAML_pred","Build_pred","AT_pred","YAML_Check","Build_Check","AT_Check"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"Missing column(s) in input: {missing}")

# Normalize to 0/1
Yp = to01(df["YAML_pred"]);  Bp = to01(df["Build_pred"]);  Ap = to01(df["AT_pred"])
Yt = to01(df["YAML_Check"]); Bt = to01(df["Build_Check"]); At = to01(df["AT_Check"])

# Ensure stratum label (from predictions if not provided)
if "stratum" in df.columns:
    strata = df["stratum"].astype(str)
else:
    strata = pd.Series([f"Y{y}_B{b}_A{a}" for y,b,a in zip(Yp,Bp,Ap)], index=df.index)

# ---------- 1) Per-stratum accuracy (exact triplet match) ----------
triplet_match = (Yp.eq(Yt) & Bp.eq(Bt) & Ap.eq(At)).astype(int)
per_stratum = (
    pd.DataFrame({"stratum": strata, "match": triplet_match})
      .groupby("stratum", dropna=False)
      .agg(n=("match","size"), matches=("match","sum"))
      .reset_index()
      .sort_values("stratum", ignore_index=True)
)
per_stratum["accuracy"] = per_stratum["matches"] / per_stratum["n"]
overall_triplet_accuracy = float(triplet_match.mean())

# ---------- 2) Per-factor precision/recall/F1 (whole sample) ----------
rows = []
for label, pred, true in [("YAML",Yp,Yt), ("Build",Bp,Bt), ("AT",Ap,At)]:
    tp, fp, fn, tn = counts_for(pred, true)
    prec, rec, f1 = pr_from_counts(tp, fp, fn)
    rows.append({
        "factor": label,
        "N": int(len(pred)),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(prec,4) if pd.notna(prec) else np.nan,
        "recall":    round(rec, 4) if pd.notna(rec) else np.nan,
        "f1":        round(f1,   4) if pd.notna(f1) else np.nan,
    })
per_factor_metrics = pd.DataFrame(rows)

# ---------- 3) OVERALL precision/recall/F1 (whole sample) ----------
# Micro-average (pool all Y,B,AT decisions):
tp = per_factor_metrics["TP"].sum()
fp = per_factor_metrics["FP"].sum()
fn = per_factor_metrics["FN"].sum()
prec_micro, rec_micro, f1_micro = pr_from_counts(tp, fp, fn)

# Macro-average (unweighted mean across factors):
prec_macro = per_factor_metrics["precision"].mean(skipna=True)
rec_macro  = per_factor_metrics["recall"].mean(skipna=True)
f1_macro   = per_factor_metrics["f1"].mean(skipna=True)

overall_metrics = pd.DataFrame([
    {"overall_type":"micro", "precision":round(prec_micro,4), "recall":round(rec_micro,4), "f1":round(f1_micro,4)},
    {"overall_type":"macro", "precision":round(prec_macro,4),  "recall":round(rec_macro,4),  "f1":round(f1_macro,4)},
    {"overall_type":"triplet_exact_accuracy", "precision":np.nan, "recall":np.nan, "f1":np.nan}
])
# Attach the triplet exact-match accuracy as a side note:
overall_note = pd.DataFrame([{"overall_triplet_exact_accuracy": round(overall_triplet_accuracy,4),
                              "rows": len(df)}])

# ---------- Save ----------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    per_stratum.to_excel(xw, sheet_name="Stratum_Accuracy", index=False)
    per_factor_metrics.to_excel(xw, sheet_name="Per_Factor_PRF1", index=False)
    overall_metrics.to_excel(xw, sheet_name="Overall_PRF1", index=False)
    overall_note.to_excel(xw, sheet_name="Overall_Note", index=False)

print(f"[OK] Saved metrics → {OUT_XLSX}")
print("Overall (micro) P/R/F1:", round(prec_micro,4), round(rec_micro,4), round(f1_micro,4))
print("Overall (macro) P/R/F1:", round(prec_macro,4), round(rec_macro,4), round(f1_macro,4))
print("Triplet exact-match accuracy:", round(overall_triplet_accuracy,4))


[OK] Saved metrics → C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx
Overall (micro) P/R/F1: 0.9855 0.9891 0.9873
Overall (macro) P/R/F1: 0.9818 0.988 0.9848
Triplet exact-match accuracy: 0.9636
